## Env setup for in-context scimilarity queries

In [1]:
# pip install tqdm if you don't already have it
from pathlib import Path
import os, gc, numpy as np, scipy.sparse as sp, torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from scimilarity.cell_annotation import CellAnnotation
from scimilarity.utils import align_dataset, lognorm_counts
from sklearn.metrics import accuracy_score, classification_report
import json
import scanpy as sc
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import sys 

sys.path.append('../code')

if 'GCAHelperFunctions' in sys.modules:
    del sys.modules['GCAHelperFunctions']


from GCAHelperFunctions import *

In [2]:
# Set up parameters
MODEL_DIR   = Path("/Users/kylekimler/Projects/flagship/scimilarity/data/model_v1.1")
H5AD_PATH   = Path("/Users/kylekimler/Projects/GCA/meta_datasets/20250717_CxG_upload_checkpoint.h5ad")
TEST_SIZE   = 0.20
RAND_SEED   = 42
BATCH = 32  # lower if kernel still crashes 

## Load and preprocess data

In [3]:

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

ca = CellAnnotation(model_path=str(MODEL_DIR))



In [4]:
# Data loading and preprocessing
data = sc.read_h5ad(H5AD_PATH)
data.layers['counts'] = data.X

if "gene_symbol" in data.var.columns:
    data.var.set_index("gene_symbol", inplace=True)

if data.var.index.has_duplicates:
    dup_mask = data.var.index.duplicated(keep="first")
    print(f"[warn] Dropping {dup_mask.sum()} duplicated gene symbols.")
    data = data[:, ~dup_mask].copy()

# Align and preprocess exactly as in scimilarity pipeline
data = align_dataset(data, ca.gene_order)

data = lognorm_counts(data)



[warn] Dropping 10 duplicated gene symbols.


In [ ]:
import obonet
import requests
from io import StringIO
import pandas as pd

tax_path = "/Users/kylekimler/Projects/GCA/ontology/gca_celltype_taxonomy.csv"

# Load GCA celltype taxonomy
taxonomy = pd.read_csv(tax_path)

# add CL term for harmonized author celltype to anndata
celltype_to_ontology = taxonomy.set_index("closest_GCA_celltype")["cell_type_ontology_term_id"].dropna()
data.obs["closest_GCA_ontology_term"] = data.obs["closest_GCA_celltype"].map(celltype_to_ontology)


# Download the latest CL ontology (only once needed)
url = 'http://purl.obolibrary.org/obo/cl/cl-basic.obo'
response = requests.get(url)
graph = obonet.read_obo(StringIO(response.text))

# Make a mapping from CL ID to name
cl_to_name = {
    node_id: data.get('name')
    for node_id, data in graph.nodes(data=True)
    if node_id.startswith('CL:')
}

# This is directly from author cell types
data.obs['groundtruth_ontology_labels'] = data.obs['cell_type_ontology_term_id'].map(cl_to_name)

data.obs['harmonized_author_ontology_labels'] = data.obs['closest_GCA_ontology_term'].map(cl_to_name)

taxonomy['harmonized_author_ontology_labels'] = taxonomy['cell_type_ontology_term_id'].map(cl_to_name)


In [ ]:
# More taxonomy setup
resolutions = ["Lineage", "Differentia", "Habitus", "Status", "Whim"]
extras = ["markers", "cell_type_ontology_term_id", "harmonized_author_ontology_labels"]
tree = build_tree_from_taxonomy(
    tax_path,
    resolution_cols=resolutions,
    annotation_cols=['harmonized_author_ontology_labels']
)

flat_labels, flat_depths = flatten( 
    tree, 
    name_key="harmonized_author_ontology_labels" 
)

plot_tree(tree)

## Subset and query-map lineages with scimilarity

### Epithelial

In [ ]:
epi_labels = taxonomy.loc[taxonomy.Lineage.eq('Epithelial'),
                             'harmonized_author_ontology_labels']\
                        .dropna().unique()

epi_labels

In [ ]:
LABEL_KEY   = "groundtruth_ontology_labels" 

data_epi = data[data.obs[LABEL_KEY].isin(epi_labels)]

labels = data_epi.obs[LABEL_KEY].astype(str).values

target_celltypes = np.unique(labels)

ca.safelist_celltypes(labels)

In [ ]:
data_epi

In [10]:
embeddings = ca.get_embeddings(data.X)

In [ ]:
# Full model Epithelium-subtype safelisted scimilarity predictions
pred_epi, _, _, _ = ca.get_predictions_knn(embeddings, weighting=True)
print("\nFull-model accuracy predicting author CL labels:", accuracy_score(labels, pred_epi))
print(classification_report(labels, pred_epi, digits=3))
